In [ ]:
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# ---------- Config ----------
DATA_DIR = Path('./data')          # folder containing your PDFs
DB_DIR = './chroma_db'             # where the vector DB is stored
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 5
EMBED_BATCH = 100
LLM_MODEL = 'gemini-2.5-flash'     # check Google's docs for current model names
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# ---------- Models and store ----------
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
db = Chroma(
    collection_name='pdfs',
    embedding_function=embeddings,
    persist_directory=DB_DIR,
)
llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)


# ---------- Indexing ----------
def already_indexed_files() -> set[str]:
    data = db.get(include=['metadatas'])
    return {m['source'] for m in data['metadatas'] if m and 'source' in m}


def index_new_pdfs() -> None:
    if not DATA_DIR.exists():
        raise SystemExit(f'Folder not found: {DATA_DIR.resolve()}')

    pdfs = sorted(DATA_DIR.rglob('*.pdf'))
    if not pdfs:
        raise SystemExit(f'No PDFs found in {DATA_DIR.resolve()}')

    done = already_indexed_files()
    new_pdfs = [p for p in pdfs if str(p) not in done]
    print(f'Found {len(pdfs)} PDFs, {len(new_pdfs)} new to index.')

    for n, pdf in enumerate(new_pdfs, 1):
        try:
            pages = PyPDFLoader(str(pdf)).load()
        except Exception as e:
            print(f'  [{n}/{len(new_pdfs)}] Skipping {pdf.name}: {e}')
            continue

        pages = [p for p in pages if p.page_content.strip()]
        if not pages:
            print(f'  [{n}/{len(new_pdfs)}] {pdf.name}: no extractable text '
                  f'(scanned? needs OCR). Skipped.')
            continue

        chunks = splitter.split_documents(pages)
        for i in range(0, len(chunks), EMBED_BATCH):
            db.add_documents(chunks[i : i + EMBED_BATCH])
        print(f'  [{n}/{len(new_pdfs)}] {pdf.name}: {len(chunks)} chunks')


# ---------- Question answering ----------
PROMPT = '''Answer the question using only the context below.
If the answer is not in the context, say you could not find it in the documents.
Cite sources in the form (filename, p.N).

Context:
{context}

Question: {question}'''


def ask(question: str) -> str:
    hits = db.similarity_search(question, k=TOP_K)
    context = '\n\n'.join(
        f'[{Path(h.metadata["source"]).name}, p.{h.metadata.get("page", 0) + 1}]\n'
        f'{h.page_content}'
        for h in hits
    )
    prompt = PROMPT.format(context=context, question=question)

    for attempt in range(4):
        try:
            return llm.invoke(prompt).content
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 15 * (attempt + 1)
                print(f'(rate limited, retrying in {wait}s)')
                time.sleep(wait)
            else:
                raise
    return 'Gemini is rate limiting requests. Try again in a minute.'


# ---------- Main ----------
if __name__ == '__main__':
    index_new_pdfs()
    print('\nReady. Ask a question (type \'exit\' to quit).\n')
    while True:
        q = input('You: ').strip()
        if q.lower() in {'exit', 'quit', ''}:
            break
        print('\n' + ask(q) + '\n')


/home/avsub/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 936.81it/s]


Found 16 PDFs, 16 new to index.
  [1/16] EviSmart_cover.pdf: 4 chunks
  [2/16] WSP_cover.pdf: 3 chunks


KeyboardInterrupt: 